# Extract Question Data from Saved Tournament HTML

**Date:** 2026-02-10  
**Purpose:** Parse saved Metaculus tournament HTML pages to extract question metadata  
**Input:** Saved HTML files in `data/`  
**Outputs:**
- `products/Question_Data_from_HTML_YYYY-MM-DD.csv` — per-question data
- `products/Tournament_Leaderboard_YYYY-MM-DD.csv` — tournament leaderboard

### Three data sources extracted from each HTML file

**Source 1 — Question Cards (visual HTML)**
- `question_number`, `title`, `slug`, `question_type`
- `community_forecast`, `community_range` (numeric IQR)
- `my_forecast`, `my_range` (numeric IQR)
- `status` — open, resolved, annulled, pending_reveal
- `resolution` — Yes/No (binary), winning option (MC), or Annulled
- `mc_options` — top community options with probabilities or resolution (MC only)
- `forecaster_count`, `comment_count`
- `tournament`

**Source 2 — "My Score" Table (embedded Next.js script data)**
- `coverage` — 0–100%, how long a forecast was active
- `score` — numeric score (populated for resolved questions)
- `question_weight` — 0.5 to 1.0

**Source 3 — Tournament Leaderboard (embedded JSON)**
- Per-user: username, is_bot, score, rank, coverage, contribution_count, prize

### IMPORTANT NOTES

**Status Field Reliability:**
- The `status` field from Source 1 (cards) is NOT always reliable
- Group questions often don't show "Resolved" text on their cards
- We infer resolved status from Source 2: if `score` exists, question is resolved
- Better approach: investigate Metaculus API for authoritative data

**Future Work:**
- Investigate Metaculus API before extensive HTML parsing work
- API may provide more reliable question metadata, status, and forecasts

In [1]:
import re
import csv
import json
import html
from pathlib import Path
from datetime import date

DATA_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/data")
OUTPUT_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products")

# List all HTML files to process — add more as needed
HTML_FILES = [
    DATA_DIR / "Spring 2026 AI Forecasting Benchmark Tournament 02-10-2026.html",
    # DATA_DIR / "MiniBench tournament file.html",  # uncomment when available
]

print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"HTML files to process: {len(HTML_FILES)}")
for f in HTML_FILES:
    exists = f.exists()
    size = f.stat().st_size if exists else 0
    print(f"  {'OK' if exists else 'MISSING'} {f.name} ({size:,} bytes)")

Data dir: C:\Users\Donni\projects\metac_bot_Spring_2026\data
Output dir: C:\Users\Donni\projects\metac_bot_Spring_2026\products
HTML files to process: 1
  OK Spring 2026 AI Forecasting Benchmark Tournament 02-10-2026.html (4,218,553 bytes)


In [2]:
# ── Source 1: Question Card Parser ─────────────────────────────────────────

def extract_tournament_name(content: str) -> str:
    """Extract tournament name from page title or heading."""
    # Try <title> tag first
    title_match = re.search(r'<title>([^<|]+)', content)
    if title_match:
        name = html.unescape(title_match.group(1).strip())
        # Remove trailing " | Metaculus" if present
        name = re.sub(r'\s*\|\s*Metaculus\s*$', '', name)
        if name and 'Tournament' in name:
            return name
    
    # Fallback: look for h1 heading
    h1_match = re.search(r'<h1[^>]*>([^<]+)</h1>', content)
    if h1_match:
        return html.unescape(h1_match.group(1).strip())
    
    return 'Unknown'


def parse_mc_options(after: str) -> tuple[list[dict], bool]:
    """Parse multiple choice options from a question card block.
    Returns (options_list, is_resolved).
    
    Open MC:     option_name + NN.N%
    Resolved MC: option_name + Yes/No
    """
    if 'MultipleChoiceTile' not in after[:500]:
        return [], False

    mc_block = after[:3000]

    # Resolved MC: option name then Yes/No in whitespace-pre div
    resolved_pattern = re.compile(
        r'<div class="resize-label min-w-0 flex-1[^"]*">\s*(.*?)</div>\s*'
        r'<div class="resize-label leading-0[^"]*">\s*'
        r'.*?<div class="whitespace-pre text-right">\s*'
        r'(Yes|No)\s*</div>',
        re.DOTALL
    )
    resolved_opts = list(resolved_pattern.finditer(mc_block))
    if resolved_opts:
        options = []
        for opt in resolved_opts:
            name = html.unescape(re.sub(r'<[^>]+>', '', opt.group(1)).strip())
            result = opt.group(2)
            options.append({'option': name, 'value': result})
        return options, True

    # Open MC: option name then percentage
    open_pattern = re.compile(
        r'<div class="resize-label min-w-0 flex-1[^"]*">\s*'
        r'(.*?)</div>\s*'
        r'<div class="resize-label flex-shrink-0[^"]*">\s*'
        r'([\d.]+%)\s*</div>',
        re.DOTALL
    )
    open_opts = list(open_pattern.finditer(mc_block))
    options = []
    for opt in open_opts:
        name = html.unescape(re.sub(r'<[^>]+>', '', opt.group(1)).strip())
        pct = opt.group(2)
        options.append({'option': name, 'value': pct})

    # Count hidden options
    others_match = re.search(r'(\d+)\s+others?</div>', mc_block)
    if others_match:
        options.append({'option': f"+{others_match.group(1)} others", 'value': ''})

    return options, False


def parse_question_cards(content: str, tournament: str) -> dict:
    """Parse all question cards from the visual HTML.
    Returns dict keyed by question_number."""
    h4_pattern = re.compile(r'<h4[^>]*>(.*?)</h4>', re.DOTALL)
    questions = {}

    for h4 in h4_pattern.finditer(content):
        title = re.sub(r'<[^>]+>', '', h4.group(1)).strip()
        title = html.unescape(re.sub(r'\s+', ' ', title))
        if not title or len(title) < 10:
            continue

        before = content[max(0, h4.start() - 500):h4.start()]
        url_match = re.search(
            r'href="https://www\.metaculus\.com/questions/(\d+)/([^"]+)"',
            before
        )
        if not url_match:
            continue

        qnum = url_match.group(1)
        slug = url_match.group(2).rstrip('/')
        if qnum in questions:
            continue

        after = content[h4.end():h4.end() + 5000]

        # ── Status & Resolution ──
        status = 'open'
        resolution = ''

        res_match = re.search(
            r'Resolved\s*</span>\s*<span[^>]*purple-800[^>]*>\s*(?:<span>\s*)?([^<]+)',
            after[:2000]
        )
        if res_match:
            status = 'resolved'
            resolution = html.unescape(res_match.group(1).strip())

        if 'Annulled' in after[:2000]:
            status = 'annulled'

        revealed_match = re.search(
            r'Revealed\s*(.*?)</template>', after[:2000], re.DOTALL
        )
        if revealed_match and status == 'open':
            status = 'pending_reveal'

        # ── Community forecast ──
        community_forecast = ''
        community_range = ''

        binary_match = re.search(
            r'font-bold[^"]*text-xl[^"]*">\s*(\d+%)\s*</span>\s*'
            r'<span[^>]*>\s*chance\s*</span>',
            after
        )
        num_match = re.search(
            r'font-bold[^"]*md:text-base">\s*\n?\s*([^<]+?)\s*</div>\s*'
            r'<div[^>]*>\s*\(([^)]+)\)\s*</div>',
            after
        )

        # ── "me:" forecast ──
        me_match = re.search(r'me:\s*<span[^>]*>\s*([^<]+?)\s*</span>', after)
        my_forecast = html.unescape(me_match.group(1).strip()) if me_match else ''

        my_range = ''
        if me_match:
            after_me = after[me_match.end():me_match.end() + 300]
            range_match = re.search(r'\(([0-9][^)]*)\)', after_me)
            if range_match:
                my_range = html.unescape(range_match.group(1).strip())

        # ── Question type ──
        mc_options, mc_resolved = parse_mc_options(after)

        if mc_options:
            q_type = 'multiple_choice'
            if mc_resolved and not resolution:
                status = 'resolved'
                winners = [o['option'] for o in mc_options if o['value'] == 'Yes']
                resolution = winners[0] if winners else ''
        elif binary_match:
            q_type = 'binary'
            community_forecast = binary_match.group(1)
        elif num_match:
            q_type = 'numeric'
            community_forecast = html.unescape(num_match.group(1).strip())
            community_range = html.unescape(num_match.group(2).strip())
        else:
            q_type = 'group'  # grouped/conditional — no card-level data

        # ── Forecaster & comment counts ──
        # Search more broadly in the card HTML
        fc_match = re.search(r'([\d,]+)\s*forecasters?', after[:5000], re.IGNORECASE)
        forecaster_count = fc_match.group(1).replace(',', '') if fc_match else ''

        cm_match = re.search(r'([\d,]+)\s*comments?', after[:5000], re.IGNORECASE)
        comment_count = cm_match.group(1).replace(',', '') if cm_match else ''

        # Format MC options as a readable string
        mc_options_str = ''
        if mc_options:
            parts = [f"{o['option']}: {o['value']}" for o in mc_options if o['value']]
            mc_options_str = ' | '.join(parts)

        questions[qnum] = {
            'question_number': qnum,
            'title': title,
            'slug': slug,
            'question_type': q_type,
            'status': status,
            'resolution': resolution,
            'community_forecast': community_forecast,
            'community_range': community_range,
            'my_forecast': my_forecast,
            'my_range': my_range,
            'mc_options': mc_options_str,
            'forecaster_count': forecaster_count,
            'comment_count': comment_count,
            'tournament': tournament,
        }

    return questions


print("Source 1 functions defined")

Source 1 functions defined


In [3]:
# ── Source 2: "My Score" Table Parser ──────────────────────────────────────

def parse_my_score_table(content: str) -> dict:
    """Extract per-question Coverage, Score, and Question Weight
    from the Next.js script data embedded in the HTML.
    Returns dict keyed by question_number."""
    script_pattern = re.compile(r'<script[^>]*>(.*?)</script>', re.DOTALL)
    scores = {}

    for s in script_pattern.finditer(content):
        script = s.group(1)
        if '/questions/' not in script:
            continue

        unescaped = script.replace('\\"', '"').replace('\\/', '/').replace('\\u0026', '&')

        q_matches = list(re.finditer(
            r'"href":"/questions/(\d+)"[^}]*"children":"([^"]+)"',
            unescaped
        ))

        for qm in q_matches:
            qnum = qm.group(1)
            title = html.unescape(qm.group(2).strip())

            after = unescaped[qm.end():qm.end() + 500]
            vals = re.findall(r'"children":"([^"]*)"', after)

            coverage = vals[0] if len(vals) > 0 else ''
            score = vals[1] if len(vals) > 1 else ''
            weight = vals[2] if len(vals) > 2 else ''

            # Normalize "-" to empty
            if coverage == '-':
                coverage = ''
            if score == '-':
                score = ''

            scores[qnum] = {
                'coverage': coverage,
                'score': score,
                'question_weight': weight,
            }

    return scores


print("Source 2 function defined")

Source 2 function defined


In [4]:
# ── Source 3: Tournament Leaderboard Parser ────────────────────────────────

def parse_leaderboard(content: str) -> list[dict]:
    """Extract the tournament leaderboard from embedded JSON in script tags.
    Returns list of leaderboard entry dicts."""
    script_pattern = re.compile(r'<script[^>]*>(.*?)</script>', re.DOTALL)
    entries = []

    for s in script_pattern.finditer(content):
        script = s.group(1)
        if '"leaderboards"' not in script:
            continue

        unescaped = script.replace('\\"', '"').replace('\\/', '/')
        unescaped = unescaped.replace('\\u0026', '&')

        # Find the JSON object containing leaderboards
        lb_match = re.search(r'\{"id":\d+,"project_id".*?"entries":\[(.*?)\]\}', unescaped, re.DOTALL)
        if not lb_match:
            continue

        # Extract the full leaderboard JSON by finding matching braces
        start = unescaped.find('{"id":')
        if start == -1:
            continue

        # Parse individual entries using a targeted regex
        entry_pattern = re.compile(
            r'\{"user":(.*?),"aggregation_method":(.*?),'
            r'"score":([\d.eE+-]+),"ci_lower":(.*?),"ci_upper":(.*?),'
            r'"rank":(\d+),"excluded":(true|false),'
            r'"show_when_excluded":(true|false),'
            r'"medal":(.*?),"prize":(.*?),'
            r'"coverage":([\d.eE+-]+),'
            r'"contribution_count":(\d+)',
            re.DOTALL
        )

        for em in entry_pattern.finditer(unescaped):
            user_json = em.group(1)
            agg = em.group(2).strip('"')

            username = ''
            is_bot = False
            if user_json != 'null':
                u_match = re.search(r'"username":"([^"]+)"', user_json)
                b_match = re.search(r'"is_bot":(true|false)', user_json)
                username = u_match.group(1) if u_match else ''
                is_bot = b_match.group(1) == 'true' if b_match else False
            else:
                username = f'[{agg}]' if agg != 'null' else '[unknown]'

            entries.append({
                'username': username,
                'is_bot': is_bot,
                'score': float(em.group(3)),
                'rank': int(em.group(6)),
                'excluded': em.group(7) == 'true',
                'coverage': float(em.group(11)),
                'contribution_count': int(em.group(12)),
                'prize': em.group(10),
            })

    entries.sort(key=lambda x: x['score'], reverse=True)
    return entries


print("Source 3 function defined")

Source 3 function defined


In [5]:
# ── Process all HTML files ─────────────────────────────────────────────────

all_questions = {}   # keyed by question_number
all_scores = {}      # keyed by question_number
all_leaderboard = [] # list of entries

for html_file in HTML_FILES:
    print(f"Processing: {html_file.name}")
    with open(html_file, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    print(f"  File size: {len(content):,} chars")

    tournament = extract_tournament_name(content)
    print(f"  Tournament: {tournament}")

    # Source 1: Question cards
    cards = parse_question_cards(content, tournament)
    for qnum, q in cards.items():
        if qnum not in all_questions:
            all_questions[qnum] = q
    print(f"  Source 1 — Question cards: {len(cards)}")

    # Source 2: My Score table
    scores = parse_my_score_table(content)
    for qnum, s in scores.items():
        if qnum not in all_scores:
            all_scores[qnum] = s
    print(f"  Source 2 — My Score rows:  {len(scores)}")

    # Source 3: Leaderboard
    lb = parse_leaderboard(content)
    if lb:
        all_leaderboard = lb  # use latest
    print(f"  Source 3 — Leaderboard:    {len(lb)} entries")
    print()

Processing: Spring 2026 AI Forecasting Benchmark Tournament 02-10-2026.html
  File size: 4,218,303 chars
  Tournament: Spring 2026 AI Forecasting Benchmark Tournament
  Source 1 — Question cards: 95
  Source 2 — My Score rows:  95
  Source 3 — Leaderboard:    0 entries



In [6]:
# ── Merge Sources 1 & 2 into combined question data ───────────────────────

merged = []
for qnum in sorted(all_questions.keys(), key=int):
    q = dict(all_questions[qnum])  # copy
    s = all_scores.get(qnum, {})
    q['coverage'] = s.get('coverage', '')
    q['score'] = s.get('score', '')
    q['question_weight'] = s.get('question_weight', '')
    
    # FIX: Infer resolved status from score presence
    # If a question has a score, it must be resolved (score only appears for resolved Qs)
    if q['score'] and q['status'] == 'open':
        q['status'] = 'resolved'
    
    merged.append(q)

# Summary stats
types = {}
statuses = {}
for q in merged:
    types[q['question_type']] = types.get(q['question_type'], 0) + 1
    statuses[q['status']] = statuses.get(q['status'], 0) + 1

has_community = sum(1 for q in merged if q['community_forecast'])
has_my = sum(1 for q in merged if q['my_forecast'])
has_score = sum(1 for q in merged if q['score'])
has_coverage = sum(1 for q in merged if q['coverage'])
has_mc_opts = sum(1 for q in merged if q['mc_options'])

print(f"Total questions:         {len(merged)}")
print(f"By type:                 {types}")
print(f"By status:               {statuses}")
print(f"With community forecast: {has_community}")
print(f"With my forecast:        {has_my}")
print(f"With MC options:         {has_mc_opts}")
print(f"With coverage:           {has_coverage}")
print(f"With score:              {has_score}")

Total questions:         95
By type:                 {'binary': 31, 'numeric': 38, 'multiple_choice': 8, 'group': 18}
By status:               {'open': 79, 'resolved': 13, 'annulled': 1, 'pending_reveal': 2}
With community forecast: 69
With my forecast:        61
With MC options:         8
With coverage:           77
With score:              10


In [7]:
# ── Display: All Questions ─────────────────────────────────────────────────

print(f"{'Q#':<8} {'Type':<7} {'Status':<10} {'Crowd':<25} {'My Fcast':<22} {'Cov':<8} {'Score':<10} {'Wt':<5} {'Title'}")
print("-" * 160)
for q in merged:
    comm = q['community_forecast']
    if q['community_range']:
        comm += f" ({q['community_range']})"
    my = q['my_forecast']
    if q['my_range']:
        my += f" ({q['my_range']})"
    print(
        f"Q{q['question_number']:<7} "
        f"{q['question_type'][:6]:<7} "
        f"{q['status']:<10} "
        f"{comm:<25} "
        f"{my:<22} "
        f"{q['coverage']:<8} "
        f"{q['score']:<10} "
        f"{q['question_weight']:<5} "
        f"{q['title'][:55]}"
    )

Q#       Type    Status     Crowd                     My Fcast               Cov      Score      Wt    Title
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Q41190   binary  open       30%                       25%                    64.4%               1.0   [PRACTICE] Will there be a positive transition to a wor
Q41191   numeri  open       >8.5B $ (8.11B - >8.5B)   >8.5B (8.11B - >8.5B)  64.5%               1.0   [PRACTICE] What will be the total domestic box office i
Q41192   multip  open                                                        64.6%               1.0   [PRACTICE] Which Party will win the next Turkish presid
Q41451   multip  resolved                                                    100.0%   4.193      1.0   Which album will win the Best Traditional Country Album
Q41454   group   resolved                                                    100.0%   -197.439

In [8]:
# ── Display: Multiple Choice Questions Detail ──────────────────────────────

mc_qs = [q for q in merged if q['question_type'] == 'multiple_choice']
print(f"Multiple choice questions: {len(mc_qs)}\n")

for q in mc_qs:
    res_str = f"  Resolution: {q['resolution']}" if q['resolution'] else ''
    print(f"Q{q['question_number']}: {q['title'][:75]}")
    print(f"  Status: {q['status']}{res_str}")
    if q['coverage']:
        print(f"  Coverage: {q['coverage']}  Score: {q['score'] or '-'}  Weight: {q['question_weight']}")
    if q['mc_options']:
        for pair in q['mc_options'].split(' | '):
            print(f"    {pair}")
    print()

Multiple choice questions: 8

Q41192: [PRACTICE] Which Party will win the next Turkish presidential election? (20
  Status: open
  Coverage: 64.6%  Score: -  Weight: 1.0
    AKP: 42.1%
    CHP: 41%
    Other: 5.4%

Q41451: Which album will win the Best Traditional Country Album Award at the 2026 G
  Status: resolved  Resolution: Ain't in it for My Health (by Zach Top)
  Coverage: 100.0%  Score: 4.193  Weight: 1.0
    Ain't in it for My Health (by Zach Top): Yes
    Oh What a Beautiful World (by Willie Nelson): No
    Dollar a Day (by Charley Crockett): No

Q41517: How many dissenting votes will there be at the January 28, 2026 Federal Ope
  Status: resolved  Resolution: 2
    2: Yes
    1: No
    0: No

Q41519: Who will win the 2026 Candidates Tournament?
  Status: open
  Coverage: 75.3%  Score: -  Weight: 1.0
    Hikaru Nakamura: 29.2%
    Fabiano Caruana: 25.7%
    Rameshbabu Praggnanandhaa: 14.8%

Q41692: How many US banks will fail from January through April 2026?
  Status: open
  

In [9]:
# ── Display: Resolved Questions ───────────────────────────────────────────

resolved = [q for q in merged if q['status'] in ('resolved', 'annulled')]
print(f"Resolved/annulled questions: {len(resolved)}\n")

print(f"{'Q#':<8} {'Type':<7} {'Status':<10} {'Resolution':<30} {'Score':<12} {'Title'}")
print("-" * 120)
for q in resolved:
    print(
        f"Q{q['question_number']:<7} "
        f"{q['question_type'][:6]:<7} "
        f"{q['status']:<10} "
        f"{q['resolution']:<30} "
        f"{q['score']:<12} "
        f"{q['title'][:50]}"
    )

Resolved/annulled questions: 14

Q#       Type    Status     Resolution                     Score        Title
------------------------------------------------------------------------------------------------------------------------
Q41451   multip  resolved   Ain't in it for My Health (by Zach Top) 4.193        Which album will win the Best Traditional Country 
Q41454   group   resolved                                  -197.439     What will be the maximum intraday value of the VIX
Q41468   group   resolved                                  -65.467      How much will Nvidia's stock price returns exceed 
Q41517   multip  resolved   2                                           How many dissenting votes will there be at the Jan
Q41521   group   resolved   No                             13.039       Will the UN General Assembly adopt a resolution co
Q41537   group   resolved                                  -4.716       What will the first reported revenues after Decemb
Q41538   group   reso

In [10]:
# ── Display: Tournament Leaderboard ───────────────────────────────────────

print(f"Leaderboard entries: {len(all_leaderboard)}\n")
print(f"{'Rank':<6} {'Score':<12} {'Cov':<8} {'Qs':<5} {'Bot':<5} {'Username'}")
print("-" * 80)
for e in all_leaderboard[:30]:
    excl = '*' if e['excluded'] else ' '
    bot = 'bot' if e['is_bot'] else ''
    print(
        f"{e['rank']:<5}{excl} "
        f"{e['score']:<12.2f} "
        f"{e['coverage']:<8.2f} "
        f"{e['contribution_count']:<5} "
        f"{bot:<5} "
        f"{e['username']}"
    )
print("\n* = excluded from prize pool")

Leaderboard entries: 0

Rank   Score        Cov      Qs    Bot   Username
--------------------------------------------------------------------------------

* = excluded from prize pool


In [11]:
# ── Write Question CSV ────────────────────────────────────────────────────

question_fieldnames = [
    'question_number', 'title', 'slug', 'question_type',
    'status', 'resolution',
    'community_forecast', 'community_range',
    'my_forecast', 'my_range',
    'mc_options',
    'coverage', 'score', 'question_weight',
    'forecaster_count', 'comment_count',
    'tournament',
]

question_csv = OUTPUT_DIR / f"Question_Data_from_HTML_{date.today()}.csv"

with open(question_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=question_fieldnames)
    writer.writeheader()
    writer.writerows(merged)

print(f"Question CSV: {question_csv.name}")
print(f"  Rows: {len(merged)}")
print(f"  Columns: {len(question_fieldnames)}")

# ── Write Leaderboard CSV ─────────────────────────────────────────────────

lb_fieldnames = [
    'rank', 'username', 'is_bot', 'excluded',
    'score', 'coverage', 'contribution_count', 'prize',
]

lb_csv = OUTPUT_DIR / f"Tournament_Leaderboard_{date.today()}.csv"

with open(lb_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=lb_fieldnames)
    writer.writeheader()
    writer.writerows(all_leaderboard)

print(f"\nLeaderboard CSV: {lb_csv.name}")
print(f"  Rows: {len(all_leaderboard)}")
print(f"  Columns: {len(lb_fieldnames)}")

Question CSV: Question_Data_from_HTML_2026-02-10.csv
  Rows: 95
  Columns: 17

Leaderboard CSV: Tournament_Leaderboard_2026-02-10.csv
  Rows: 0
  Columns: 8
